# Sub-question 2: task counts versus time

> Does weighting tasks by time rather than counting them equally change the
> picture of exposure?

Same three occupations, sources and joins as `subq1.ipynb`. The only new step
is counting each exposure category two ways:

- **by tasks** -- every task counts once
- **by time** -- every task counts by its estimated hours per day

Two chart types follow. Figure A puts all three occupations side by side.
Figure B takes one occupation at a time and shows every task.

In [ ]:
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIGS = Path("figures/subq2")
FIGS.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---- the three occupations and their tasks ---------------------------------
# Same sources and joins as subq1.ipynb.
#   hours : estimated hours per day on the task        Hatgis-Kessell et al. (2026)
#   E     : E0 / E1 / E2, human-rated                  Eloundou et al.
STUDY = ["21-1093",   # Social and Human Service Assistants
         "21-1021",   # Child, Family, and School Social Workers
         "11-9151"]   # Social and Community Service Managers
DAY = 7.0             # hours; the day the hours source allocates across tasks

hours = pd.read_excel("data/raw/time_share/task_time_share_estimates.xlsx")
hours.columns = ["task_id", "task", "onet_code", "occ_name",
                 "hours", "hours_per_instance", "instances_per_day"]
hours["OCC_CODE"] = hours.onet_code.str.split(".").str[0]

elo = pd.read_csv("data/raw/gpts_are_gpts/full_labelset.tsv", sep="\t",
                  usecols=["Task ID", "human_exposure_agg"])

three = (hours[hours.OCC_CODE.isin(STUDY)]
         .merge(elo, left_on="task_id", right_on="Task ID", how="left")
         .rename(columns={"human_exposure_agg": "E"}))
TITLES = three.groupby("OCC_CODE").occ_name.first().to_dict()

assert three.E.notna().all(), "every task should carry an exposure label"
assert np.allclose(three.groupby("OCC_CODE").hours.sum(), DAY)
print(f"{len(three)} tasks across {three.OCC_CODE.nunique()} occupations")

## Tasks and hours in each exposure category

In [ ]:
# ---- tasks and hours in each exposure category -----------------------------
# Counting tasks: every task counts once.
# Counting time:  every task counts by its estimated hours per day.
TIERS = ["E1", "E2", "E0"]
E_COLOR = {"E0": "#d9dade", "E2": "#4e9e6a", "E1": "#1b4d2a"}
E_LABEL = {"E1": "Directly exposed",
           "E2": "Exposed with additional software",
           "E0": "Not exposed"}

g = three.groupby(["OCC_CODE", "E"])
summary = pd.DataFrame({"tasks": g.size(), "hours": g.hours.sum()}).reset_index()
summary["pct_tasks"] = 100 * summary.tasks / summary.groupby("OCC_CODE").tasks.transform("sum")
summary["pct_time"] = 100 * summary.hours / DAY

table = (summary.pivot(index="OCC_CODE", columns="E",
                       values=["tasks", "pct_tasks", "hours", "pct_time"])
         .reindex(STUDY).reindex(columns=TIERS, level=1))
table.index = [TITLES[c] for c in table.index]
table.round(2)

## Figure A: tasks versus time, all three occupations

Two bars per occupation. If counting tasks and counting hours told the same
story, each pair of bars would look alike.

In [ ]:
# ---- Figure A: all three occupations, tasks versus time --------------------
# Two bars per occupation, each adding to 100%. The upper bar counts tasks
# equally; the lower bar weights them by hours. Segment labels give the
# percentage, and the underlying count or hours.
def text_color(tier):
    return "white" if tier in ("E1", "E2") else "#333"


fig, ax = plt.subplots(figsize=(10, 6.2))
yticks, ylabels = [], []
y = 0.0
for code in STUDY:
    s = summary[summary.OCC_CODE == code].set_index("E").reindex(TIERS)
    for row, (pct, raw, fmt) in enumerate([("pct_tasks", "tasks", "{:.0f}"),
                                           ("pct_time", "hours", "{:.2f} h")]):
        left = 0.0
        for tier in TIERS:
            width = s.loc[tier, pct]
            ax.barh(y, width, left=left, height=0.62, color=E_COLOR[tier],
                    edgecolor="white", linewidth=1.2)
            label = f"{width:.0f}%"
            if width >= 14:
                label += f"  ({fmt.format(s.loc[tier, raw])})"
            if width >= 5:
                ax.text(left + width / 2, y, label, ha="center", va="center",
                        fontsize=8, color=text_color(tier))
            left += width
        yticks.append(y)
        ylabels.append("Task-Weighted" if row == 0 else "Time-Weighted")
        y -= 0.72
    ax.text(0, y + 2 * 0.72 + 0.46, f"{TITLES[code]}  ({len(three[three.OCC_CODE == code])} tasks)",
            fontsize=10, ha="left", va="bottom")
    y -= 1.35          # gap between occupations

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("Share of the job exposed to AI (%)", fontsize=9)
handles = [plt.Rectangle((0, 0), 1, 1, color=E_COLOR[t]) for t in TIERS]
ax.legend(handles, [E_LABEL[t] for t in TIERS], title="Theoretical Exposure",
          title_fontsize=8, fontsize=8, frameon=False,
          loc="upper right", bbox_to_anchor=(1.0, -0.06))
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.tight_layout()
fig.savefig(FIGS / "tasks_vs_time_all.png", dpi=300, bbox_inches="tight")
plt.show()

## Figure B: one dumbbell figure per occupation

How far each exposure category moves when tasks are weighted by time instead
of counted equally. A long line means the two ways of counting disagree.

In [ ]:
# ---- Figure B: one dumbbell figure per occupation --------------------------
# Three rows per occupation, one per exposure category. The grey dot is the
# category's share of the job when every task counts once; the green dot is
# its share when tasks are weighted by hours. The line between them is how far
# the category moves, labelled in percentage points.
C_TASK, C_TIME, C_LINE = "#8a97a6", "#1b4d2a", "#c9ced6"
XMAX = 70             # shared across all three figures


def detail_figure(code, fname):
    s = summary[summary.OCC_CODE == code].set_index("E").reindex(TIERS)

    fig, ax = plt.subplots(figsize=(8.5, 3.1))
    for y, tier in zip([2, 1, 0], TIERS):
        a, b = s.loc[tier, "pct_tasks"], s.loc[tier, "pct_time"]
        ax.plot([a, b], [y, y], color=C_LINE, linewidth=2.5, zorder=1,
                solid_capstyle="round")
        ax.scatter(a, y, s=70, color=C_TASK, edgecolor="white", linewidth=1, zorder=2)
        ax.scatter(b, y, s=70, color=C_TIME, edgecolor="white", linewidth=1, zorder=3)
        d = b - a
        sign = "+" if d >= 0 else "−"
        ax.text(max(a, b) + 1.8, y, f"{sign}{abs(d):.0f} {'pt' if round(abs(d)) == 1 else 'pts'}", va="center",
                fontsize=8, color=C_TIME if d >= 0 else "#5d6b7a")

    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels([E_LABEL[t] for t in TIERS], fontsize=9)
    ax.set_ylim(-0.6, 2.6)
    ax.set_xlim(0, XMAX)
    ax.set_xlabel("Share of the job (%)", fontsize=9)
    ax.set_title(TITLES[code], loc="left", fontsize=11)
    handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=7,
                          color=c, markeredgecolor="white") for c in (C_TASK, C_TIME)]
    ax.legend(handles, ["Task-Weighted", "Time-Weighted"], frameon=False,
              fontsize=8, loc="upper right", bbox_to_anchor=(1.0, -0.36), ncol=2)
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    plt.tight_layout()
    fig.savefig(FIGS / fname, dpi=300, bbox_inches="tight")
    plt.show()


for i, code in enumerate(STUDY, start=1):
    detail_figure(code, f"detail_occ{i:02d}.png")